# Análisis de los datos

Se cubren los siguientes puntos:  
1. Número de registros y estructura general  
2. Tipos de datos  
3. Valores faltantes
4. Valores únicos por variable
5. Duplicados exactos
6. Formatos inconsistentes
7. Tablas / resúmenes consolidados (exportados para el Code Book y el plan de limpieza)

In [65]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [12]:
df = pd.read_csv('../data/processed/centros_educativos_gt.csv', dtype=str, encoding="utf-8")


df.head(5)


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


## Estructura del dataset

In [30]:
filas, columnas = df.shape

print(f"Número de registros del dataset: {filas} \n")
print(f"Número de variables del dataset: {columnas} \n")
print(f"Nombre de las variables: {list(df.columns)} \n")

print(f"Tipo de datos por variable:\n{df.dtypes} \n")

print(f"Departamentos: {df['DEPARTAMENTO'].nunique()} \n")
print(f"Municipios: {df['MUNICIPIO'].nunique()} \n")
print(f"Cantidad de establecimientos por departamento: {df['DEPARTAMENTO'].value_counts()} \n")


Número de registros del dataset: 11889 

Número de variables del dataset: 17 

Nombre de las variables: ['CODIGO', 'DISTRITO', 'DEPARTAMENTO', 'MUNICIPIO', 'ESTABLECIMIENTO', 'DIRECCION', 'TELEFONO', 'SUPERVISOR', 'DIRECTOR', 'NIVEL', 'SECTOR', 'AREA', 'STATUS', 'MODALIDAD', 'JORNADA', 'PLAN', 'DEPARTAMENTAL'] 

Tipo de datos por variable:
CODIGO             str
DISTRITO           str
DEPARTAMENTO       str
MUNICIPIO          str
ESTABLECIMIENTO    str
DIRECCION          str
TELEFONO           str
SUPERVISOR         str
DIRECTOR           str
NIVEL              str
SECTOR             str
AREA               str
STATUS             str
MODALIDAD          str
JORNADA            str
PLAN               str
DEPARTAMENTAL      str
dtype: object 

Departamentos: 24 

Municipios: 353 

Cantidad de establecimientos por departamento: DEPARTAMENTO
CIUDAD CAPITAL    2161
GUATEMALA         1908
SAN MARCOS         724
ESCUINTLA          708
HUEHUETENANGO      591
QUETZALTENANGO     551
PETEN          

## Valores faltantes

Se revisan dos tipos de valores faltantes:
- **Nulos reales:** Celdas faltantes en el CSV.
- **Placeholders:** Representación de datos faltantes pero no estan vacíos.

In [38]:
PLACEHOLDERS_NULOS = ["--", "-", "N/A", "NA", "S/N", "S/D", "", " ", "NINGUNO", "NO TIENE"]

def cuenta_placeholders(serie):
    return serie.dropna().astype(str).str.strip().str.upper().isin(PLACEHOLDERS_NULOS).sum()

resumen_na = pd.DataFrame({
    "n_nulos_reales": df.isna().sum(),
    "n_placeholders": [cuenta_placeholders(df[c]) for c in df.columns],
})
resumen_na["n_faltantes_total"] = resumen_na["n_nulos_reales"] + resumen_na["n_placeholders"]
resumen_na["pct_faltantes_total"] = (resumen_na["n_faltantes_total"] / len(df) * 100).round(2)
resumen_na = resumen_na.sort_values("pct_faltantes_total", ascending=False)
resumen_na


,n_nulos_reales,n_placeholders,n_faltantes_total,pct_faltantes_total
DIRECTOR,1732,58,1790,15.06
TELEFONO,946,0,946,7.96
SUPERVISOR,535,0,535,4.50
DISTRITO,532,0,532,4.47
DIRECCION,76,1,77,0.65
ESTABLECIMIENTO,5,0,5,0.04
CODIGO,0,0,0,0.00
MUNICIPIO,0,0,0,0.00
DEPARTAMENTO,0,0,0,0.00
NIVEL,0,0,0,0.00


## Valores Únicos


In [39]:
valores_unicos = pd.DataFrame({
    "valores_unicos" : df.nunique(),
    "Porcentaje: ": (df.nunique() / len(df) * 100).round(2)
})
valores_unicos

,valores_unicos,Porcentaje:
CODIGO,11868,99.82
DISTRITO,1682,14.15
DEPARTAMENTO,24,0.20
MUNICIPIO,353,2.97
ESTABLECIMIENTO,6313,53.10
DIRECCION,7440,62.58
TELEFONO,6572,55.28
SUPERVISOR,1281,10.77
DIRECTOR,5517,46.40
NIVEL,2,0.02


## Valores Duplicados exactos
Se analiza los registros duplicados analizando:
- Filas completamente duplicadas
- Filas duplicadas sin tomar en cuenta el `CODIGO` para revisar si algun establecimiento fue registrado mas de una vez
- Posibles establecimientos repetidos donde los datos mas relevantes para identificar un establecimiento estan repetidos. 

In [64]:
print(f"Filas completamente duplicadas: {df.duplicated().sum()}")

print(f"\nFilas duplicadas sin considerar codigo: {df.drop(columns=['CODIGO']).duplicated().sum()}")

codigo_duplicado = df[df["CODIGO"].duplicated(keep=False)]

print(f"\nCódigos distintos repetidos: {codigo_duplicado['CODIGO'].nunique()}")
print(f"Filas involucradas: {len(codigo_duplicado)}")

if len(codigo_duplicado) > 0:
    display(codigo_duplicado.sort_values("CODIGO"))

establecimientos_repetidos =df[df.duplicated(
    subset=["ESTABLECIMIENTO","MUNICIPIO","DEPARTAMENTO","DIRECCION"],
    keep=False
)]

display(establecimientos_repetidos.sort_values(
    ["ESTABLECIMIENTO","MUNICIPIO"]
))

Filas completamente duplicadas: 21

Filas duplicadas sin considerar codigo: 221

Códigos distintos repetidos: 1
Filas involucradas: 22


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
475,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
11165,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
10972,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
10758,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
10033,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
9602,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
9237,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
8914,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
8362,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
7845,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
8028,17-05-0211-46,17-05-0034,PETEN,LA LIBERTAD,"""CENTRO DE EDUCACION INTEGRAL EL NARANJO""",ALDEA EL NARANJO,57283598,ARIEL ESAU PELAEZ,MIRIÁN VIRGINIA MARTÍNEZ CANTÉ,DIVERSIFICADO,PRIVADO,RURAL,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),PETÉN
8046,17-05-0260-46,17-05-0034,PETEN,LA LIBERTAD,"""CENTRO DE EDUCACION INTEGRAL EL NARANJO""",ALDEA EL NARANJO,57283598,ARIEL ESAU PELAEZ,MIRIÁN VIRGINIA MARTÍNEZ CANTÉ,DIVERSIFICADO,PRIVADO,RURAL,ABIERTA,BILINGUE,DOBLE,FIN DE SEMANA,PETÉN
8066,17-05-0319-46,17-05-0034,PETEN,LA LIBERTAD,"""CENTRO DE EDUCACION INTEGRAL EL NARANJO""",ALDEA EL NARANJO,53663885,ARIEL ESAU PELAEZ,MIRIÁN VIRGINIA MARTÍNEZ CANTÉ,DIVERSIFICADO,PRIVADO,RURAL,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),PETÉN
8073,17-05-0335-46,17-05-0034,PETEN,LA LIBERTAD,"""CENTRO DE EDUCACION INTEGRAL EL NARANJO""",ALDEA EL NARANJO,53663885,ARIEL ESAU PELAEZ,MIRIÁN VIRGINIA MARTÍNEZ CANTÉ,DIVERSIFICADO,PRIVADO,RURAL,ABIERTA,MONOLINGUE,SIN JORNADA,SEMIPRESENCIAL (UN DÍA A LA SEMANA),PETÉN
3943,05-02-1922-46,05-02-0267,ESCUINTLA,SANTA LUCIA COTZUMALGUAPA,"""CENTRO EDUCATIVO INGENIO LA UNIÓN""","KM. 98.8 CARRETERA A INGENIO LOS TARROS, FINCA...",77401593,JUAN FRANCISCO GODOY DÁVILA,EDUARDO NOÉ ESTRADA SANTIZO,DIVERSIFICADO,PRIVADO,RURAL,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),ESCUINTLA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10284,12-07-0166-46,12-090,SAN MARCOS,TACANA,TECNOLOGICO SPENCER W. KIMBALL,COLONIA LA DEMOCRACIA,30819832,FREDY ARMANDO FELICIANO GABRIEL,RICARDO ESTEBAN CIFUENTES ORTIZ,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,DOBLE,DIARIO(REGULAR),SAN MARCOS
10285,12-07-0167-46,12-090,SAN MARCOS,TACANA,TECNOLOGICO SPENCER W. KIMBALL,COLONIA LA DEMOCRACIA,30819832,FREDY ARMANDO FELICIANO GABRIEL,RICARDO ESTEBAN CIFUENTES ORTIZ,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),SAN MARCOS
828,04-01-3066-46,04-01-0219,CHIMALTENANGO,CHIMALTENANGO,TECNOLÓGICO PREUNIVERSITARIO,3ERA. CALLE 7-54 ZONA 3,41502786,MILDRED JEANNTTE ABAJ MAZATE,GERSON ARELY PUZ SAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,DOBLE,FIN DE SEMANA,CHIMALTENANGO
829,04-01-3067-46,04-01-0219,CHIMALTENANGO,CHIMALTENANGO,TECNOLÓGICO PREUNIVERSITARIO,3ERA. CALLE 7-54 ZONA 3,41502786,MILDRED JEANNTTE ABAJ MAZATE,GERSON ARELY PUZ SAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),CHIMALTENANGO


## Formatos inconsistentes

Se revisan los patrones típicos:
- Espacios extra (al inicio/final o dobles internos)
- Formato de `CODIGO` (se espera `DD-DD-NNNN-NN`)
- Formato de `DISTRITO` (se observan al menos dos patrones distintos en los datos de ejemplo)
- Formato de `TELEFONO` (se espera 8 dígitos numéricos)
- Inconsistencias de mayúsculas/minúsculas en columnas categóricas
- Caracteres especiales embebidos en nombres de establecimientos (comillas, siglas entre guiones)
- Coherencia entre `DEPARTAMENTO` y `DEPARTAMENTAL` (deberían ser siempre iguales)

In [72]:
hallazgos = []

for col in df.columns:
    n = df[col].apply(lambda x: isinstance(x, str) and x != x.strip()).sum()
    if n > 0: 
        hallazgos.append((col, "espacios al inicio o final", n))

for col in df.columns:
    n = df[col].astype(str).str.contains(r"\s{2,}", regex=True, na=False).sum()
    if n > 0:
        hallazgos.append((col, "espacios dobles", n))


patron_codigo = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
codigo_invalido = (~df["CODIGO"].astype(str).str.match(patron_codigo).sum())
hallazgos.append(("CODIGO", "formato inválido", codigo_invalido))

def clasifica_distrito(x):
    if pd.isna(x):
        return "FALTANTE"
    if re.match(r"^\d{2}-\d{3}$", x):
        return "DD-NNN"
    if re.match(r"^\d{2}-\d{2}-\d{4}$", x):
        return "DD-DD-NNNN"
    return "OTRO"

patrones_distrito = df["DISTRITO"].apply(clasifica_distrito)
print("Patrones distintos encontrados en DISTRITO:")
print(patrones_distrito.value_counts())
hallazgos.append(("DISTRITO", "múltiples patrones de formato (ver detalle arriba)", patrones_distrito.nunique()))


patron_telefono = re.compile(r"^\d{8}$")
telefono = df["TELEFONO"].dropna().astype(str).str.strip()
telefono = telefono[telefono != ""]
telefono_invalido = (~telefono.str.match(patron_telefono).sum())
hallazgos.append(("TELEFONO", "formato inválido", telefono_invalido))


for col in ["DEPARTAMENTO", "MUNICIPIO", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]:
    valores = df[col].dropna().astype(str)
    no_mayusculas = (valores != valores.str.upper()).sum()
    if no_mayusculas > 0:
        hallazgos.append((col, "no estan en mayuscula", no_mayusculas))


n_comillas = df["ESTABLECIMIENTO"].astype(str).str.contains(r'"').sum()
hallazgos.append(("ESTABLECIMIENTO", "contiene comillas dobles", n_comillas))

n_guion_final = df["ESTABLECIMIENTO"].astype(str).str.contains(r"-\w+-\s*$").sum()
hallazgos.append(("ESTABLECIMIENTO", "termina en sigla entre guiones, ej. -IIAV-", n_guion_final))

n_mismatch = (df["DEPARTAMENTO"].astype(str).str.strip() != df["DEPARTAMENTAL"].astype(str).str.strip()).sum()
hallazgos.append(("DEPARTAMENTO vs DEPARTAMENTAL", "no coinciden entre sí", n_mismatch))

hallazgos_df = pd.DataFrame(hallazgos, columns=["Variable", "Inconsistencia", "Registros_afectados"])
hallazgos_df = hallazgos_df[hallazgos_df["Registros_afectados"] > 0].sort_values("Registros_afectados", ascending=False)
hallazgos_df


Patrones distintos encontrados en DISTRITO:
DISTRITO
DD-DD-NNNN    6226
DD-NNN        5039
FALTANTE       532
OTRO            92
Name: count, dtype: int64


,Variable,Inconsistencia,Registros_afectados
5,DEPARTAMENTO vs DEPARTAMENTAL,no coinciden entre sí,6116
3,ESTABLECIMIENTO,contiene comillas dobles,2228
4,ESTABLECIMIENTO,"termina en sigla entre guiones, ej. -IIAV-",265
1,DISTRITO,múltiples patrones de formato (ver detalle arr...,4
